In [7]:
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import timedelta

from joblib import dump
from sklearn.preprocessing import StandardScaler

from statsmodels.tsa.stattools import adfuller

In [8]:
# dataset requirements
# timestamp index
# individual label for local series
# variable cols (including 1+ contextual optimizer, e.g. cost)
# target col

dir = './dummy_generator/'
fname = 'dummy_'
fform = '.csv'
fpath = dir + fname + fform
df0 = (pd.read_csv(fpath)).drop_duplicates()
df0.head()

,timestamp,index,0,1,2,3,4,5,6,7,...,23,24,25,26,27,28,29,class,bc_target_sqrt,bc_adj_log_optimizer
0,1997-01-05,43,3.213688,-1.660000,-2.759641,-3.213148,5.508263,4.138808,-2.948606,0.645985,...,4.736792,-1.485509,0.134723,-5.101437,-0.617050,3.606649,-6.721429,19,14337.134417,57.260197
1,1997-01-05,79,3.118646,2.305061,-3.270608,4.299180,2.411507,0.222985,2.114374,-4.702906,...,4.246429,-2.184006,-5.341166,-2.930877,-3.449490,4.681910,-9.479493,9,12750.035889,64.762961
2,1997-01-05,91,-2.118167,4.298264,-4.294911,1.101183,-4.822803,1.161821,-2.016493,-1.547995,...,-0.199345,-5.667066,-0.549626,-0.643603,11.604386,-5.002744,-3.650987,2,28135.258734,55.349736
3,1997-01-05,257,-3.139244,2.396480,-1.207759,-0.828921,0.871641,4.404288,-4.123759,2.212615,...,1.034566,0.516768,4.736482,-3.066393,-7.066498,0.718723,-16.872716,32,15712.381757,57.868667
4,1997-01-05,428,-6.924534,-2.881754,-4.770541,-2.029026,0.532607,-1.379486,2.603288,-3.483671,...,1.242792,5.020350,-0.568237,-4.237205,-8.556563,2.717400,-3.014155,9,14356.729918,58.524822


In [9]:
# flag variables
ts_index = 'timestamp'                    #timestamp index column name
l_label = 'index'                         #local series identifier
variate_cols = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21',
       '22', '23', '24', '25', '26', '27', '28', '29', 'class',
       'bc_adj_log_optimizer']            #characterizer variables column names
cont_factors = ['bc_adj_log_optimizer']   #contextual factor(s) column names
target_col = 'bc_target_sqrt'             #prediction target column names

#relevant value columns
v_cols = list(set(cont_factors + [target_col]))

In [10]:
# ensure timestamp formating
dtf = '%Y-%m-%d' #string format of datetime index
df0[ts_index] = pd.to_datetime(df0[ts_index], format=dtf)
df0.sort_values(ts_index, ascending=True, inplace=True)

# ensure cont_factors and target are moved last
new_cols = [col for col in df0.columns if col not in v_cols] + v_cols
df0 = df0[new_cols]

In [11]:
# transformation step to accomodate different stationarity states locally

## run augmented dickey-fuller test to control stationarity
def adf_t(col_series, pv=0.05, verbose=False):    
#        col_series = column with series for test
#        pv = p-val threshold
#        autolag = akaike's information criteria
    r = adfuller(col_series, autolag='AIC')
    output = {'stat test':round(r[0], 4), 'pval':round(r[1], 4), 'lags':round(r[2], 4), 'obs':r[3]}
    pv = output['pval']     
    if r[1] <= pv:
        #stationary
        st = 1
    else:
        #non-stationary
        st = 0
    return st

In [12]:
#transform one local series at a time
df_diff = pd.DataFrame()
st_check = 0

#pre-difference base df
df_base = df0.copy()

#max differencing order
md_ord = 6

In [13]:
#iterate over local series
for i in list(set(df_base[l_label])):

    #temporary local df
    df_l = df_base[df_base[l_label]==i].copy()

    df_l_i = df_l[[c for c in df_base.columns if c \
        not in v_cols]].copy()

    df_l_d = df_l[v_cols].copy()

    #local series st check
    st_check = 1
    
    #difference order
    diff_order = 0
    df_l_d['diff_order'] = 0 #placeholder

    while st_check < 2:        
           
        c = 0
        
        #iterate over contextual factors and target col
        while c < len(v_cols):
            
            # column to be checked
            col = v_cols[c]    

            # non stat first
            st_i = 0

            #try adf, assume stationary if failed
            # (will occur for very short, cold-starts, etc.)
            try:
                st_i = adf_t(df_l_d[col])
                if st_i == 1:
                    st_check = 1
                    c = c + 1
            except:
                st_i = 1
                st_check = 1
                c = c + 1
            
            if st_i == 0:
                st_check = 0
                c = 0
                df_l_d = df_l_d.diff().dropna()
                diff_order = diff_order + 1
            
        if st_check == 1 or diff_order > md_ord:

            #if stat after all cols are checked, break cycle
            df_l = pd.concat([df_l_i, df_l_d], axis=1)
            
            #move target to last
            df_l = df_l.reindex(columns = [col for col in df_l.columns \
                                           if col != target_col] + [target_col])
            
            df_l['diff_order'] = diff_order
            
            # continuous append
            df_diff = pd.concat([df_diff, df_l], axis=0, ignore_index=True)  
            st_check = 2
            #break

In [14]:
((df_diff.dropna()).drop_duplicates()).to_csv('diff_' + fname + '.csv', index=False)

In [15]:
df_diff.head()

,timestamp,index,0,1,2,3,4,5,6,7,...,24,25,26,27,28,29,class,bc_adj_log_optimizer,diff_order,bc_target_sqrt
0,1997-06-08,0,1.354563,-2.403948,3.561855,0.527249,-1.228229,-3.703629,-1.03211,1.5959,...,3.347139,3.062488,-2.018713,-8.507892,-0.424906,13.874133,37,NaN,38,NaN
1,1997-06-15,0,1.354563,-2.403948,3.561855,0.527249,-1.228229,-3.703629,-1.03211,1.5959,...,3.347139,3.062488,-2.018713,-8.507892,-0.424906,13.874133,37,NaN,38,NaN
2,1997-06-22,0,1.354563,-2.403948,3.561855,0.527249,-1.228229,-3.703629,-1.03211,1.5959,...,3.347139,3.062488,-2.018713,-8.507892,-0.424906,13.874133,37,NaN,38,NaN
3,1997-06-29,0,1.354563,-2.403948,3.561855,0.527249,-1.228229,-3.703629,-1.03211,1.5959,...,3.347139,3.062488,-2.018713,-8.507892,-0.424906,13.874133,37,NaN,38,NaN
4,1997-07-06,0,1.354563,-2.403948,3.561855,0.527249,-1.228229,-3.703629,-1.03211,1.5959,...,3.347139,3.062488,-2.018713,-8.507892,-0.424906,13.874133,37,NaN,38,NaN


In [16]:
# data scaling
scaler = StandardScaler()
scaler.fit(((df_diff[[c for c in \
                df_diff.columns if c != l_label]]).set_index(ts_index)).to_numpy())

dump(scaler, 'scaler_' + fname + '.joblib')

np.save('scaler_stats_' + fname + '.npy', np.array([scaler.mean_, scaler.scale_]))

['scaler_dummy_.joblib']